<a href="https://colab.research.google.com/github/Karthikreddy1010/MicroscopicPlastic-Semantic-segmentation/blob/main/three_seed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!unzip /content/archive.zip

In [ ]:
!pip install tensorflow

In [6]:
"""
================================================================================
3-SEED REPRODUCIBILITY EXPERIMENT
================================================================================
Runs the SAME four segmentation models (exactly as validated in the four
ablation notebooks) across three random seeds (42, 123, 2024) to determine
whether the differences observed in the 4-way ablation are reproducible.

NOTHING about the architectures, dataset, preprocessing, augmentation,
losses, optimizer settings, threshold-selection methodology, or evaluation
metrics has been changed. This script only adds a seed loop + bookkeeping
around the exact training/eval protocol already used in:
    ablation_efficientnetv2b3_baseline.ipynb
    ablation_2_efficientnetv2b3_mrfm.ipynb
    ablation_3_efficientnetv2b3_attention.ipynb
    ablation_4_full_proposed_model.ipynb

HOW TO RUN
----------
This trains 4 models x up to 3 seeds x ~2-phase training with a real
EfficientNetV2-B3 backbone at 512x512. It needs a GPU (Colab/local),
internet access once to download ImageNet weights, and your dataset.
It is NOT executed here — copy it into your own environment (e.g. a
Colab notebook cell, or `python three_seed_reproducibility_experiment.py`)
and run it there.

BEFORE RUNNING
--------------
1. Set DATA_ROOT below to the parent folder that already contains the
   exact same train/val/test subfolders your four ablation notebooks used.
   Do NOT re-split the data — this script loads the same fixed directories,
   so the split is identical for every seed and every model automatically.
2. (Optional) If you still have the four `results_efficientnetv2b3_*/results.csv`
   files produced by the original notebooks, keep them next to this script
   (or set SEED42_RESULTS_CSV_MAP below) so the seed-42 TRAIN rows can be
   pulled in automatically. Otherwise only the seed-42 TEST reference values
   given in your prompt are available, and TRAIN will be left blank for seed 42.
3. Seed 42 is NOT retrained by default (INCLUDE_SEED_42_RETRAIN = False) —
   its results are the existing reference. Set the flag to True only if you
   explicitly want to re-verify seed 42 as well.
================================================================================
"""

import os
import re
import glob
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models, backend as K
from tensorflow.keras import mixed_precision
from tensorflow.keras.applications import EfficientNetV2B3
from tensorflow.keras.applications.efficientnet_v2 import preprocess_input
from PIL import Image
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.utils.class_weight import compute_class_weight
from skimage.transform import rotate, resize
from scipy import stats

warnings.filterwarnings('ignore')

# ==============================================================================
# CONFIGURATION — MUST MATCH THE ORIGINAL ABLATION NOTEBOOKS EXACTLY
# ==============================================================================

# >>> SET THIS to the verified dataset root already used in your ablation code.
# >>> Do NOT point this at /content and do NOT change the subfolder structure
# >>> below — it must resolve to the exact same images used previously so the
# >>> train/val/test split is identical across every seed and model.
#DATA_ROOT = "/path/to/your/verified/dataset"

TRAIN_IMAGE_DIR = "/content/training/training/Training_original"
TRAIN_MASK_DIR = "/content/training/training/Finalmasks"
VAL_IMAGE_DIR = "/content/val/val/Original"
VAL_MASK_DIR = "/content/val/val/val_100masks"
TEST_IMAGE_DIR = "/content/Testing/Testing/Original"
TEST_MASK_DIR = "/content/Testing/Testing/testing_100masks"

IMG_HEIGHT = 512
IMG_WIDTH = 512
BATCH_SIZE = 4
EPOCHS = 100          # Phase 2 max epochs (same as original)
PHASE1_EPOCHS = 40    # Phase 1 epochs (same as original)
PATIENCE = 15

THRESHOLD_RANGE = np.linspace(0.1, 0.7, 13)  # identical search range/steps

SEEDS = [123, 2024]
INCLUDE_SEED_42_RETRAIN = False  # set True to also retrain seed 42 from scratch

RESULTS_ROOT = "results"
CHECKPOINT_ROOT = "results"  # results/seed_X/model_key/*.weights.h5
OUTPUT_DIR = "three_seed_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

MODEL_ORDER = [
    "EfficientNetV2-B3 Baseline",
    "EfficientNetV2-B3 + MRFM",
    "EfficientNetV2-B3 + Stage-Adaptive Attention",
    "EfficientNetV2-B3 + MRFM + Stage-Adaptive Attention",
]

MODEL_KEYS = {
    "EfficientNetV2-B3 Baseline": "baseline",
    "EfficientNetV2-B3 + MRFM": "mrfm",
    "EfficientNetV2-B3 + Stage-Adaptive Attention": "attention",
    "EfficientNetV2-B3 + MRFM + Stage-Adaptive Attention": "proposed",
}

# Existing seed-42 reference (TEST only, exactly as given — do not overwrite).
SEED42_TEST_REFERENCE = {
    "EfficientNetV2-B3 Baseline": dict(
        Dice=0.785754, Foreground_IoU=0.647112, Background_IoU=0.991925,
        mIoU=0.819519, Precision=0.759414, Recall=0.813985),
    "EfficientNetV2-B3 + MRFM": dict(
        Dice=0.787396, Foreground_IoU=0.649343, Background_IoU=0.991970,
        mIoU=0.820657, Precision=0.759536, Recall=0.817378),
    "EfficientNetV2-B3 + Stage-Adaptive Attention": dict(
        Dice=0.785372, Foreground_IoU=0.646595, Background_IoU=0.991770,
        mIoU=0.819182, Precision=0.747261, Recall=0.827580),
    "EfficientNetV2-B3 + MRFM + Stage-Adaptive Attention": dict(
        Dice=0.786139, Foreground_IoU=0.647636, Background_IoU=0.991853,
        mIoU=0.819745, Precision=0.752455, Recall=0.822981),
}

# If you kept the original per-model results.csv files (each has a TRAIN and
# a TEST row), point to them here so seed-42 TRAIN rows can be recovered too.
SEED42_RESULTS_CSV_MAP = {
    "EfficientNetV2-B3 Baseline": "results_efficientnetv2b3_baseline/results.csv",
    "EfficientNetV2-B3 + MRFM": "results_efficientnetv2b3_mrfm_only/results.csv",
    "EfficientNetV2-B3 + Stage-Adaptive Attention": "results_efficientnetv2b3_attention_only/results.csv",
    "EfficientNetV2-B3 + MRFM + Stage-Adaptive Attention": "results_efficientnetv2b3_full_proposed/results.csv",
}


# ==============================================================================
# GPU / MIXED PRECISION SETUP (identical to originals)
# ==============================================================================

def configure_gpu():
    gpus = tf.config.list_physical_devices('GPU')
    if len(gpus) > 0:
        print(" GPU detected:", gpus)
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        mixed_precision.set_global_policy('mixed_float16')
        print(" Mixed precision enabled")
    else:
        print(" No GPU detected - running on CPU (this will be very slow)")


def set_global_seed(seed):
    """Propagate one seed to every stochastic component used anywhere
    in the pipeline: Python random, NumPy, TensorFlow, dataset shuffling,
    model initialization, and augmentation randomness."""
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    print(f" Global seed set to {seed} (python random / numpy / tensorflow)")


# ==============================================================================
# DATA UTILITIES (IDENTICAL TO ORIGINAL — unchanged from the ablation notebooks)
# ==============================================================================

def normalize_name(name):
    """Normalize filename for matching."""
    name = os.path.basename(name)
    name = os.path.splitext(name)[0]
    name = name.replace('_mask', '')
    return name


def preprocess_mask(mask):
    """Proper mask binarization."""
    mask_binary = (mask > 127).astype(np.float32)
    return mask_binary


def calculate_iou(y_true, y_pred):
    """Calculate Intersection over Union for binary masks."""
    y_true = y_true.flatten().astype(np.uint8)
    y_pred = y_pred.flatten().astype(np.uint8)
    intersection = np.sum(y_true * y_pred)
    union = np.sum(y_true) + np.sum(y_pred) - intersection
    if union == 0:
        return 0.0
    return intersection / union


def calculate_iou_metric(y_true, y_pred):
    """Standard IoU metric."""
    y_true_flat = y_true.flatten().astype(np.uint8)
    y_pred_flat = y_pred.flatten().astype(np.uint8)
    intersection = np.sum(y_true_flat * y_pred_flat)
    union = np.sum(y_true_flat) + np.sum(y_pred_flat) - intersection
    if union == 0:
        return 1.0 if intersection == 0 else 0.0
    return intersection / union


def calculate_iou_per_image(y_true, y_pred):
    """Calculate IoU for each image individually."""
    ious = []
    for i in range(len(y_true)):
        gt = y_true[i].flatten().astype(np.uint8)
        pred = y_pred[i].flatten().astype(np.uint8)
        intersection = np.sum(gt * pred)
        union = np.sum(gt) + np.sum(pred) - intersection
        if union == 0:
            iou = 1.0 if intersection == 0 else 0.0
        else:
            iou = intersection / union
        ious.append(iou)
    return np.array(ious)


def calculate_comprehensive_iou_metrics(y_true, y_pred):
    """Calculate comprehensive IoU metrics."""
    y_true_flat = y_true.flatten().astype(np.uint8)
    y_pred_flat = y_pred.flatten().astype(np.uint8)

    iou_bg = calculate_iou(1 - y_true_flat, 1 - y_pred_flat)
    iou_fg = calculate_iou(y_true_flat, y_pred_flat)
    miou = (iou_bg + iou_fg) / 2.0

    iou_standard = calculate_iou_metric(y_true, y_pred)
    per_image_ious = calculate_iou_per_image(y_true, y_pred)

    return {
        'iou': iou_standard,
        'iou_background': iou_bg,
        'iou_foreground': iou_fg,
        'miou': miou,
        'iou_mean': np.mean(per_image_ious),
        'iou_std': np.std(per_image_ious),
        'iou_median': np.median(per_image_ious),
        'iou_min': np.min(per_image_ious),
        'iou_max': np.max(per_image_ious),
        'per_image_ious': per_image_ious
    }


def calculate_segmentation_metrics(y_true, y_pred_binary):
    """Calculate pixel-wise segmentation metrics."""
    y_true_flat = y_true.flatten().astype(np.uint8)
    y_pred_flat = y_pred_binary.flatten().astype(np.uint8)

    precision = precision_score(y_true_flat, y_pred_flat, zero_division=0)
    recall = recall_score(y_true_flat, y_pred_flat, zero_division=0)
    f1 = f1_score(y_true_flat, y_pred_flat, zero_division=0)

    intersection = np.sum(y_true_flat * y_pred_flat)
    dice = (2. * intersection) / (np.sum(y_true_flat) + np.sum(y_pred_flat) + 1e-8)

    iou_metrics = calculate_comprehensive_iou_metrics(y_true, y_pred_binary)

    return {
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'dice': dice,
        'iou': iou_metrics['iou'],
        'iou_background': iou_metrics['iou_background'],
        'iou_foreground': iou_metrics['iou_foreground'],
        'miou': iou_metrics['miou'],
        'iou_mean': iou_metrics['iou_mean'],
        'iou_std': iou_metrics['iou_std'],
        'iou_median': iou_metrics['iou_median'],
        'iou_min': iou_metrics['iou_min'],
        'iou_max': iou_metrics['iou_max']
    }


# ==============================================================================
# AUGMENTATION PIPELINE (IDENTICAL TO ORIGINAL)
# ==============================================================================

def augment_geometric_numpy(image, mask, seed=None):
    """Apply geometric augmentations (rotation, zoom, flip) with consistent randomness."""
    if hasattr(image, 'numpy'):
        image = image.numpy()
    if hasattr(mask, 'numpy'):
        mask = mask.numpy()

    image = np.asarray(image, dtype=np.float32)
    mask = np.asarray(mask, dtype=np.float32)

    if seed is not None:
        np.random.seed(seed)

    h, w = image.shape[:2]

    if np.random.rand() < 0.5:
        angle = np.random.uniform(-15, 15)
        image = rotate(image, angle, mode='reflect', preserve_range=True)
        mask = rotate(mask, angle, mode='reflect', preserve_range=True)

    if np.random.rand() < 0.5:
        zoom_factor = np.random.uniform(0.9, 1.1)
        new_h, new_w = int(h * zoom_factor), int(w * zoom_factor)

        image_zoomed = resize(image, (new_h, new_w), preserve_range=True, anti_aliasing=True)
        mask_zoomed = resize(mask, (new_h, new_w), preserve_range=True, anti_aliasing=False, order=0)

        if zoom_factor > 1.0:
            start_h = (new_h - h) // 2
            start_w = (new_w - w) // 2
            image = image_zoomed[start_h:start_h+h, start_w:start_w+w]
            mask = mask_zoomed[start_h:start_h+h, start_w:start_w+w]
        else:
            pad_h = (h - new_h) // 2
            pad_w = (w - new_w) // 2

            if len(mask_zoomed.shape) == 3:
                image = np.pad(image_zoomed, ((pad_h, h-new_h-pad_h), (pad_w, w-new_w-pad_w), (0, 0)),
                              mode='reflect')
                mask = np.pad(mask_zoomed, ((pad_h, h-new_h-pad_h), (pad_w, w-new_w-pad_w), (0, 0)),
                             mode='constant', constant_values=0)
            else:
                image = np.pad(image_zoomed, ((pad_h, h-new_h-pad_h), (pad_w, w-new_w-pad_w), (0, 0)),
                              mode='reflect')
                mask = np.pad(mask_zoomed, ((pad_h, h-new_h-pad_h), (pad_w, w-new_w-pad_w)),
                             mode='constant', constant_values=0)

    if np.random.rand() < 0.5:
        image = np.fliplr(image)
        mask = np.fliplr(mask)

    if np.random.rand() < 0.5:
        image = np.flipud(image)
        mask = np.flipud(mask)

    if len(mask.shape) == 2:
        mask = np.expand_dims(mask, axis=-1)

    return image.astype(np.float32), mask.astype(np.float32)


def augment_photometric_numpy(image, seed=None):
    """Apply photometric augmentations to image only."""
    if hasattr(image, 'numpy'):
        image = image.numpy()

    image = np.asarray(image, dtype=np.float32)

    if seed is not None:
        np.random.seed(seed)

    brightness = np.random.uniform(-0.05, 0.05)
    image = image + brightness

    contrast = np.random.uniform(0.95, 1.05)
    mean = image.mean()
    image = (image - mean) * contrast + mean

    gamma = np.random.uniform(0.95, 1.05)
    image = np.power(np.clip(image, 0, 1), gamma)

    noise = np.random.normal(0, 0.005, image.shape).astype(np.float32)
    image = image + noise

    image = np.clip(image, 0, 1)

    return image.astype(np.float32)


def augment_combined_numpy(image, mask):
    """Combined augmentation pipeline."""
    if hasattr(image, 'numpy'):
        image = image.numpy()
    if hasattr(mask, 'numpy'):
        mask = mask.numpy()

    image = np.asarray(image, dtype=np.float32)
    mask = np.asarray(mask, dtype=np.float32)

    seed = np.random.randint(0, 10000)

    image, mask = augment_geometric_numpy(image, mask, seed=seed)
    image = augment_photometric_numpy(image, seed=seed + 100)

    return image, mask


def load_data_raw(image_dir, mask_dir):
    """Load RAW data without preprocessing - returns [0, 1] range."""
    print(f"Loading RAW data from: {image_dir}")

    if not os.path.exists(image_dir):
        raise ValueError(f"Image directory does not exist: {image_dir}")
    if not os.path.exists(mask_dir):
        raise ValueError(f"Mask directory does not exist: {mask_dir}")

    image_paths = sorted([
        os.path.join(image_dir, fname)
        for fname in os.listdir(image_dir)
        if fname.lower().endswith(('.png', '.jpg', '.jpeg'))
    ])

    print(f"Found {len(image_paths)} images")

    images = []
    masks = []

    for img_idx, img_path in enumerate(image_paths):
        fname = os.path.basename(img_path)
        fname_key = normalize_name(fname)

        mask_found = False
        mask_path = None

        mask_candidate = os.path.join(mask_dir, fname_key + '_mask.png')
        if os.path.exists(mask_candidate):
            mask_path = mask_candidate
            mask_found = True

        if not mask_found:
            for ext in ['.png', '.jpg', '.jpeg', '.PNG', '.JPG', '.JPEG']:
                mask_candidate = os.path.join(mask_dir, fname_key + ext)
                if os.path.exists(mask_candidate):
                    mask_path = mask_candidate
                    mask_found = True
                    break

        if not mask_found:
            base_name = os.path.splitext(fname)[0]
            for ext in ['.png', '.jpg', '.jpeg', '.PNG', '.JPG', '.JPEG']:
                mask_candidate = os.path.join(mask_dir, base_name + ext)
                if os.path.exists(mask_candidate):
                    mask_path = mask_candidate
                    mask_found = True
                    break

        if not mask_found or mask_path is None:
            if img_idx < 5:
                print(f"Warning: No mask found for {fname_key}")
            continue

        try:
            img = Image.open(img_path).convert('RGB')
            img = img.resize((IMG_WIDTH, IMG_HEIGHT), Image.BILINEAR)
            img_array = np.array(img, dtype=np.float32) / 255.0

            mask = Image.open(mask_path).convert('L')
            mask = mask.resize((IMG_WIDTH, IMG_HEIGHT), Image.NEAREST)
            mask_array = np.array(mask, dtype=np.float32)
            mask_array = preprocess_mask(mask_array)
            mask_array = np.expand_dims(mask_array, axis=-1)

            images.append(img_array)
            masks.append(mask_array)

        except Exception as e:
            print(f"Error loading {fname}: {e}")
            continue

    if len(images) == 0:
        raise ValueError(f"No valid images loaded from {image_dir}")

    images = np.array(images, dtype=np.float32)
    masks = np.array(masks, dtype=np.float32)

    print(f"Successfully loaded {len(images)} images")
    return images, masks


def create_augmented_dataset_with_preprocessing(images, masks, batch_size, augment=True, shuffle=True):
    """Create tf.data.Dataset with augmentation THEN preprocessing."""

    def preprocess_for_efficientnet(img, mask):
        img = img * 255.0
        img = preprocess_input(img)
        return img, mask

    dataset = tf.data.Dataset.from_tensor_slices((images, masks))

    if shuffle:
        dataset = dataset.shuffle(buffer_size=len(images), reshuffle_each_iteration=True)

    if augment:
        def augment_fn(img, mask):
            img, mask = tf.py_function(
                func=augment_combined_numpy,
                inp=[img, mask],
                Tout=[tf.float32, tf.float32]
            )
            img.set_shape([IMG_HEIGHT, IMG_WIDTH, 3])
            mask.set_shape([IMG_HEIGHT, IMG_WIDTH, 1])
            return img, mask

        dataset = dataset.map(augment_fn, num_parallel_calls=tf.data.AUTOTUNE)

    dataset = dataset.map(preprocess_for_efficientnet, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)

    return dataset


def preprocess_test_data(X):
    """Preprocess data the same way as training data (no augmentation)."""
    return preprocess_input(X * 255.0)


# ==============================================================================
# LOSS FUNCTIONS & METRICS (IDENTICAL TO ORIGINAL)
# ==============================================================================

def calculate_class_weights(masks):
    """Calculate class weights for imbalanced segmentation."""
    masks_flat = (masks > 0.5).astype(np.uint8).flatten()
    class_weights = compute_class_weight(
        class_weight='balanced',
        classes=np.array([0, 1]),
        y=masks_flat
    )
    return {0: class_weights[0], 1: class_weights[1]}


def dice_coefficient(y_true, y_pred, smooth=1e-6):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    y_true_flat = tf.reshape(y_true, [-1])
    y_pred_flat = tf.reshape(y_pred, [-1])
    intersection = tf.reduce_sum(y_true_flat * y_pred_flat)
    union = tf.reduce_sum(y_true_flat) + tf.reduce_sum(y_pred_flat)
    return (2.0 * intersection + smooth) / (union + smooth)


def dice_loss(y_true, y_pred, smooth=1e-6):
    return 1.0 - dice_coefficient(y_true, y_pred, smooth)


def focal_loss(y_true, y_pred, alpha=0.25, gamma=2.0):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    y_true_flat = tf.reshape(y_true, [-1])
    y_pred_flat = tf.reshape(y_pred, [-1])
    y_pred_flat = tf.clip_by_value(y_pred_flat, K.epsilon(), 1 - K.epsilon())

    pt = tf.where(tf.equal(y_true_flat, 1), y_pred_flat, 1 - y_pred_flat)
    alpha_t = tf.where(tf.equal(y_true_flat, 1), alpha, 1 - alpha)
    loss = -alpha_t * tf.pow(1 - pt, gamma) * tf.math.log(pt)
    return tf.reduce_mean(loss)


def tversky_loss(y_true, y_pred, alpha=0.7, beta=0.3, smooth=1e-6):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    y_true_flat = tf.reshape(y_true, [-1])
    y_pred_flat = tf.reshape(y_pred, [-1])

    true_pos = tf.reduce_sum(y_true_flat * y_pred_flat)
    false_neg = tf.reduce_sum(y_true_flat * (1 - y_pred_flat))
    false_pos = tf.reduce_sum((1 - y_true_flat) * y_pred_flat)

    tversky = (true_pos + smooth) / (true_pos + alpha * false_neg + beta * false_pos + smooth)
    return 1.0 - tversky


def aggressive_imbalance_loss(y_true, y_pred):
    """Combined loss for severe class imbalance (Phase 1 loss)."""
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    y_pred = tf.clip_by_value(y_pred, K.epsilon(), 1 - K.epsilon())

    dice = dice_loss(y_true, y_pred)
    tversky = tversky_loss(y_true, y_pred, alpha=0.7, beta=0.3)
    focal = focal_loss(y_true, y_pred, alpha=0.75, gamma=2.0)

    return 5.0 * dice + 3.0 * tversky + 1.0 * focal


def focal_dice_loss(y_true, y_pred, gamma=2.0, dice_weight=3.0, focal_weight=1.0):
    """Combined focal and dice loss (Phase 2 loss)."""
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    focal = focal_loss(y_true, y_pred, alpha=0.75, gamma=gamma)
    dice = dice_loss(y_true, y_pred)
    return focal_weight * focal + dice_weight * dice


def iou_metric(y_true, y_pred, smooth=1e-6):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    y_true_flat = tf.reshape(y_true, [-1])
    y_pred_flat = tf.reshape(y_pred, [-1])
    intersection = tf.reduce_sum(y_true_flat * y_pred_flat)
    union = tf.reduce_sum(y_true_flat) + tf.reduce_sum(y_pred_flat) - intersection
    return (intersection + smooth) / (union + smooth)


def dice_numpy(y_true, y_pred, smooth=1e-6):
    y_true = y_true.astype(np.float32).flatten()
    y_pred = y_pred.astype(np.float32).flatten()
    intersection = np.sum(y_true * y_pred)
    return (2. * intersection + smooth) / (np.sum(y_true) + np.sum(y_pred) + smooth)


class PredictionMonitor(tf.keras.callbacks.Callback):
    """Monitor predictions during training (identical to original)."""
    def __init__(self, val_data, val_masks):
        super().__init__()
        self.val_data_raw = val_data[:4]
        self.val_masks = val_masks[:4]
        self.val_data = preprocess_input(self.val_data_raw * 255.0)

    def on_epoch_end(self, epoch, logs=None):
        if epoch % 3 == 0:
            preds = self.model.predict(self.val_data, verbose=0)
            preds_flat = preds.flatten()
            fg_pixels = (preds_flat > 0.5).sum()
            total_pixels = preds_flat.size
            print(f"   [Epoch {epoch}] max={preds_flat.max():.4f}, mean={preds_flat.mean():.4f}, "
                  f"fg_pixels={fg_pixels:,}/{total_pixels:,} ({fg_pixels/total_pixels*100:.3f}%)")


# ==============================================================================
# SHARED ARCHITECTURE MODULES (MRFM + Stage-Adaptive Attention)
# Reused verbatim across the models that use them, exactly as in the
# ablation notebooks.
# ==============================================================================

def MRFM(x, filters, name):
    """Multi-Resolution Fusion Module. Output channels = filters // 2."""
    out_ch = filters // 2
    shortcut = x

    b3 = layers.Conv2D(filters, 3, padding='same', kernel_initializer='he_normal', name=f'{name}_b3')(x)
    b3 = layers.BatchNormalization(name=f'{name}_b3_bn')(b3)
    b3 = layers.Activation('relu', name=f'{name}_b3_relu')(b3)

    b5 = layers.Conv2D(filters, 5, padding='same', kernel_initializer='he_normal', name=f'{name}_b5')(x)
    b5 = layers.BatchNormalization(name=f'{name}_b5_bn')(b5)
    b5 = layers.Activation('relu', name=f'{name}_b5_relu')(b5)

    bd = layers.Conv2D(filters, 3, dilation_rate=2, padding='same', kernel_initializer='he_normal', name=f'{name}_bd')(x)
    bd = layers.BatchNormalization(name=f'{name}_bd_bn')(bd)
    bd = layers.Activation('relu', name=f'{name}_bd_relu')(bd)

    fused = layers.Concatenate(name=f'{name}_concat')([b3, b5, bd])
    fused = layers.Conv2D(out_ch, 1, padding='same', kernel_initializer='he_normal', name=f'{name}_compress')(fused)
    fused = layers.BatchNormalization(name=f'{name}_compress_bn')(fused)
    fused = layers.Activation('relu', name=f'{name}_compress_relu')(fused)

    if K.int_shape(shortcut)[-1] != out_ch:
        shortcut = layers.Conv2D(out_ch, 1, padding='same', kernel_initializer='he_normal', name=f'{name}_shortcut')(shortcut)
        shortcut = layers.BatchNormalization(name=f'{name}_shortcut_bn')(shortcut)

    out = layers.Add(name=f'{name}_add')([fused, shortcut])
    return layers.Activation('relu', name=f'{name}_final_relu')(out)


class EdgeAttention(layers.Layer):
    """Edge-aware attention for high-resolution early stages."""
    def __init__(self, channels, **kwargs):
        super().__init__(**kwargs)
        self.channels = channels

    def build(self, input_shape):
        self.edge_conv = layers.Conv2D(self.channels, 3, padding='same', activation='relu',
                                        kernel_initializer='he_normal', name=f'{self.name}_edge_detector')
        self.attention_conv = layers.Conv2D(self.channels, 1, padding='same', activation='sigmoid',
                                             kernel_initializer='glorot_uniform', name=f'{self.name}_edge_attention')
        super().build(input_shape)

    def call(self, x):
        edges = self.edge_conv(x)
        attention = self.attention_conv(edges)
        return x * attention

    def get_config(self):
        config = super().get_config()
        config.update({'channels': self.channels})
        return config


class CoordinateAttention(layers.Layer):
    """Coordinate attention for middle stages."""
    def __init__(self, channels, reduction=8, **kwargs):
        super().__init__(**kwargs)
        self.channels = channels
        self.reduction = reduction

    def build(self, input_shape):
        self.conv1 = layers.Conv2D(self.channels // self.reduction, 1, activation='relu',
                                    kernel_initializer='he_normal', name=f'{self.name}_coord_reduce')
        self.conv_h = layers.Conv2D(self.channels, 1, activation='sigmoid',
                                     kernel_initializer='glorot_uniform', name=f'{self.name}_coord_h')
        self.conv_w = layers.Conv2D(self.channels, 1, activation='sigmoid',
                                     kernel_initializer='glorot_uniform', name=f'{self.name}_coord_w')
        super().build(input_shape)

    def call(self, x):
        shape = tf.shape(x)
        height = shape[1]
        width = shape[2]

        pool_h = tf.reduce_mean(x, axis=2, keepdims=True)
        pool_w = tf.reduce_mean(x, axis=1, keepdims=True)

        pool_w_t = tf.transpose(pool_w, [0, 2, 1, 3])
        concat = tf.concat([pool_h, pool_w_t], axis=1)

        concat = self.conv1(concat)

        split_h, split_w = tf.split(concat, [height, width], axis=1)
        split_w = tf.transpose(split_w, [0, 2, 1, 3])

        att_h = self.conv_h(split_h)
        att_w = self.conv_w(split_w)

        return x * att_h * att_w

    def get_config(self):
        config = super().get_config()
        config.update({'channels': self.channels, 'reduction': self.reduction})
        return config


class ChannelSpatialAttention(layers.Layer):
    """Combined channel and spatial attention for deep stages."""
    def __init__(self, channels, reduction=8, **kwargs):
        super().__init__(**kwargs)
        self.channels = channels
        self.reduction = reduction

    def build(self, input_shape):
        self.global_avg_pool = layers.GlobalAveragePooling2D(keepdims=False)
        self.global_max_pool = layers.GlobalMaxPooling2D(keepdims=False)

        self.fc1 = layers.Dense(self.channels // self.reduction, activation='relu',
                                 kernel_initializer='he_normal', name=f'{self.name}_channel_fc1')
        self.fc2 = layers.Dense(self.channels, activation='sigmoid',
                                 kernel_initializer='glorot_uniform', name=f'{self.name}_channel_fc2')

        self.spatial_conv = layers.Conv2D(1, 7, padding='same', activation='sigmoid',
                                           kernel_initializer='glorot_uniform', name=f'{self.name}_spatial_attention')
        super().build(input_shape)

    def call(self, x):
        avg_pool = self.global_avg_pool(x)
        max_pool = self.global_max_pool(x)

        avg_out = self.fc2(self.fc1(avg_pool))
        max_out = self.fc2(self.fc1(max_pool))

        channel_att = avg_out + max_out
        channel_att = tf.reshape(channel_att, [-1, 1, 1, self.channels])
        x_channel = x * channel_att

        avg_spatial = tf.reduce_mean(x_channel, axis=-1, keepdims=True)
        max_spatial = tf.reduce_max(x_channel, axis=-1, keepdims=True)
        spatial_concat = tf.concat([avg_spatial, max_spatial], axis=-1)

        spatial_att = self.spatial_conv(spatial_concat)
        return x_channel * spatial_att

    def get_config(self):
        config = super().get_config()
        config.update({'channels': self.channels, 'reduction': self.reduction})
        return config


class MicroplasticAttention(layers.Layer):
    """Stage-adaptive attention: Edge (early) / Coordinate (middle) / Channel-Spatial (deep)."""
    def __init__(self, channels, stage='early', **kwargs):
        super().__init__(**kwargs)
        self.channels = channels
        self.stage = stage

    def build(self, input_shape):
        if self.stage == 'early':
            self.attention = EdgeAttention(self.channels, name=f'{self.name}_edge')
        elif self.stage == 'middle':
            self.attention = CoordinateAttention(self.channels, name=f'{self.name}_coord')
        elif self.stage == 'deep':
            self.attention = ChannelSpatialAttention(self.channels, name=f'{self.name}_chan_spat')
        else:
            raise ValueError(f"Invalid stage: {self.stage}")
        super().build(input_shape)

    def call(self, x):
        return self.attention(x)

    def get_config(self):
        config = super().get_config()
        config.update({'channels': self.channels, 'stage': self.stage})
        return config


def _extract_encoder_features(backbone):
    e1 = backbone.get_layer('block1b_add').output
    e2 = backbone.get_layer('block2b_add').output
    e3 = backbone.get_layer('block3b_add').output
    e4 = backbone.get_layer('block5c_add').output
    e5 = backbone.get_layer('top_activation').output
    return e1, e2, e3, e4, e5


def _final_decoder_tail(d1):
    """Identical final upsampling tail used by all four models."""
    d0 = layers.Conv2DTranspose(16, 2, strides=2, padding='same',
                                 kernel_initializer='he_normal', name='up0')(d1)
    d0 = layers.Conv2D(32, 3, padding='same', activation='relu',
                        kernel_initializer='he_normal', name='final_conv1')(d0)
    d0 = layers.Conv2D(16, 3, padding='same', activation='relu',
                        kernel_initializer='he_normal', name='final_conv2')(d0)
    outputs = layers.Conv2D(1, 1, activation='sigmoid', dtype='float32',
                             kernel_initializer='glorot_uniform', name='segmentation_output')(d0)
    return outputs


# ------------------------------------------------------------------------------
# MODEL 1: BASELINE (no MRFM, no attention)
# ------------------------------------------------------------------------------
def build_efficientnetv2b3_baseline_unet(input_shape=(512, 512, 3)):
    inputs = layers.Input(shape=input_shape, name='input_image')
    backbone = EfficientNetV2B3(include_top=False, weights='imagenet', input_tensor=inputs)
    backbone.trainable = True

    e1, e2, e3, e4, e5 = _extract_encoder_features(backbone)

    def decoder_block(x, skip, num_filters, name):
        x = layers.Conv2DTranspose(num_filters, 2, strides=2, padding='same',
                                    kernel_initializer='he_normal', name=f'{name}_up')(x)
        x = layers.Concatenate(name=f'{name}_concat')([x, skip])
        x = layers.Conv2D(num_filters, 3, padding='same', activation='relu',
                           kernel_initializer='he_normal', name=f'{name}_conv1')(x)
        x = layers.BatchNormalization(name=f'{name}_bn1')(x)
        x = layers.Conv2D(num_filters, 3, padding='same', activation='relu',
                           kernel_initializer='he_normal', name=f'{name}_conv2')(x)
        x = layers.BatchNormalization(name=f'{name}_bn2')(x)
        return x

    d4 = decoder_block(e5, e4, 256, 'dec4')
    d3 = decoder_block(d4, e3, 128, 'dec3')
    d2 = decoder_block(d3, e2, 64, 'dec2')
    d1 = decoder_block(d2, e1, 32, 'dec1')

    outputs = _final_decoder_tail(d1)
    model = models.Model(inputs, outputs, name='EfficientNetV2B3_Baseline_UNet')
    return model, backbone


# ------------------------------------------------------------------------------
# MODEL 2: + MRFM (no attention)
# ------------------------------------------------------------------------------
def build_efficientnetv2b3_mrfm_unet(input_shape=(512, 512, 3)):
    inputs = layers.Input(shape=input_shape, name='input_image')
    backbone = EfficientNetV2B3(include_top=False, weights='imagenet', input_tensor=inputs)
    backbone.trainable = True

    e1, e2, e3, e4, e5 = _extract_encoder_features(backbone)

    e1p = MRFM(e1, 64, 'mrfm1')
    e2p = MRFM(e2, 128, 'mrfm2')
    e3p = MRFM(e3, 256, 'mrfm3')
    e4p = MRFM(e4, 512, 'mrfm4')
    e5p = MRFM(e5, 512, 'mrfm5')

    d4 = layers.Conv2DTranspose(256, 2, strides=2, padding='same', kernel_initializer='he_normal', name='up4')(e5p)
    d4 = layers.Concatenate(name='concat4')([d4, e4p])
    d4 = MRFM(d4, 512, 'dec4')

    d3 = layers.Conv2DTranspose(128, 2, strides=2, padding='same', kernel_initializer='he_normal', name='up3')(d4)
    d3 = layers.Concatenate(name='concat3')([d3, e3p])
    d3 = MRFM(d3, 256, 'dec3')

    d2 = layers.Conv2DTranspose(64, 2, strides=2, padding='same', kernel_initializer='he_normal', name='up2')(d3)
    d2 = layers.Concatenate(name='concat2')([d2, e2p])
    d2 = MRFM(d2, 128, 'dec2')

    d1 = layers.Conv2DTranspose(32, 2, strides=2, padding='same', kernel_initializer='he_normal', name='up1')(d2)
    d1 = layers.Concatenate(name='concat1')([d1, e1p])
    d1 = MRFM(d1, 64, 'dec1')

    outputs = _final_decoder_tail(d1)
    model = models.Model(inputs, outputs, name='EfficientNetV2B3_MRFM_UNet')
    return model, backbone


# ------------------------------------------------------------------------------
# MODEL 3: + Stage-Adaptive Attention (no MRFM)
# ------------------------------------------------------------------------------
def build_efficientnetv2b3_attention_unet(input_shape=(512, 512, 3)):
    inputs = layers.Input(shape=input_shape, name='input_image')
    backbone = EfficientNetV2B3(include_top=False, weights='imagenet', input_tensor=inputs)
    backbone.trainable = True

    e1, e2, e3, e4, e5 = _extract_encoder_features(backbone)
    e1c, e2c, e3c, e4c, e5c = (K.int_shape(t)[-1] for t in (e1, e2, e3, e4, e5))

    e1p = MicroplasticAttention(e1c, stage='early', name='enc1_attn')(e1)
    e2p = MicroplasticAttention(e2c, stage='early', name='enc2_attn')(e2)
    e3p = MicroplasticAttention(e3c, stage='middle', name='enc3_attn')(e3)
    e4p = MicroplasticAttention(e4c, stage='deep', name='enc4_attn')(e4)
    e5p = MicroplasticAttention(e5c, stage='deep', name='enc5_attn')(e5)

    def decoder_block(x, skip, num_filters, name):
        x = layers.Conv2DTranspose(num_filters, 2, strides=2, padding='same',
                                    kernel_initializer='he_normal', name=f'{name}_up')(x)
        x = layers.Concatenate(name=f'{name}_concat')([x, skip])
        x = layers.Conv2D(num_filters, 3, padding='same', activation='relu',
                           kernel_initializer='he_normal', name=f'{name}_conv1')(x)
        x = layers.BatchNormalization(name=f'{name}_bn1')(x)
        x = layers.Conv2D(num_filters, 3, padding='same', activation='relu',
                           kernel_initializer='he_normal', name=f'{name}_conv2')(x)
        x = layers.BatchNormalization(name=f'{name}_bn2')(x)
        return x

    d4 = decoder_block(e5p, e4p, 256, 'dec4')
    d4 = MicroplasticAttention(256, stage='deep', name='dec4_attn')(d4)

    d3 = decoder_block(d4, e3p, 128, 'dec3')
    d3 = MicroplasticAttention(128, stage='middle', name='dec3_attn')(d3)

    d2 = decoder_block(d3, e2p, 64, 'dec2')
    d2 = MicroplasticAttention(64, stage='early', name='dec2_attn')(d2)

    d1 = decoder_block(d2, e1p, 32, 'dec1')
    d1 = MicroplasticAttention(32, stage='early', name='dec1_attn')(d1)

    outputs = _final_decoder_tail(d1)
    model = models.Model(inputs, outputs, name='EfficientNetV2B3_StageAdaptiveAttention_UNet')
    return model, backbone


# ------------------------------------------------------------------------------
# MODEL 4: + MRFM + Stage-Adaptive Attention (full proposed model)
# ------------------------------------------------------------------------------
def build_efficientnetv2b3_mrfm_attention_unet(input_shape=(512, 512, 3)):
    inputs = layers.Input(shape=input_shape, name='input_image')
    backbone = EfficientNetV2B3(include_top=False, weights='imagenet', input_tensor=inputs)
    backbone.trainable = True

    e1, e2, e3, e4, e5 = _extract_encoder_features(backbone)

    e1p = MRFM(e1, 64, 'mrfm1')
    e1p = MicroplasticAttention(32, stage='early', name='enc1_attn')(e1p)

    e2p = MRFM(e2, 128, 'mrfm2')
    e2p = MicroplasticAttention(64, stage='early', name='enc2_attn')(e2p)

    e3p = MRFM(e3, 256, 'mrfm3')
    e3p = MicroplasticAttention(128, stage='middle', name='enc3_attn')(e3p)

    e4p = MRFM(e4, 512, 'mrfm4')
    e4p = MicroplasticAttention(256, stage='deep', name='enc4_attn')(e4p)

    e5p = MRFM(e5, 512, 'mrfm5')
    e5p = MicroplasticAttention(256, stage='deep', name='enc5_attn')(e5p)

    d4 = layers.Conv2DTranspose(256, 2, strides=2, padding='same', kernel_initializer='he_normal', name='up4')(e5p)
    d4 = layers.Concatenate(name='concat4')([d4, e4p])
    d4 = MRFM(d4, 512, 'dec4')
    d4 = MicroplasticAttention(256, stage='deep', name='dec4_attn')(d4)

    d3 = layers.Conv2DTranspose(128, 2, strides=2, padding='same', kernel_initializer='he_normal', name='up3')(d4)
    d3 = layers.Concatenate(name='concat3')([d3, e3p])
    d3 = MRFM(d3, 256, 'dec3')
    d3 = MicroplasticAttention(128, stage='middle', name='dec3_attn')(d3)

    d2 = layers.Conv2DTranspose(64, 2, strides=2, padding='same', kernel_initializer='he_normal', name='up2')(d3)
    d2 = layers.Concatenate(name='concat2')([d2, e2p])
    d2 = MRFM(d2, 128, 'dec2')
    d2 = MicroplasticAttention(64, stage='early', name='dec2_attn')(d2)

    d1 = layers.Conv2DTranspose(32, 2, strides=2, padding='same', kernel_initializer='he_normal', name='up1')(d2)
    d1 = layers.Concatenate(name='concat1')([d1, e1p])
    d1 = MRFM(d1, 64, 'dec1')
    d1 = MicroplasticAttention(32, stage='early', name='dec1_attn')(d1)

    outputs = _final_decoder_tail(d1)
    model = models.Model(inputs, outputs, name='EfficientNetV2B3_MRFM_StageAdaptiveAttention_UNet')
    return model, backbone


MODEL_BUILDERS = {
    "EfficientNetV2-B3 Baseline": build_efficientnetv2b3_baseline_unet,
    "EfficientNetV2-B3 + MRFM": build_efficientnetv2b3_mrfm_unet,
    "EfficientNetV2-B3 + Stage-Adaptive Attention": build_efficientnetv2b3_attention_unet,
    "EfficientNetV2-B3 + MRFM + Stage-Adaptive Attention": build_efficientnetv2b3_mrfm_attention_unet,
}


# ==============================================================================
# ONE TRAIN + EVAL RUN FOR A GIVEN (SEED, MODEL)
# ==============================================================================

def train_and_evaluate_one(seed, model_name, X_train, y_train, X_val, y_val, X_test, y_test):
    model_key = MODEL_KEYS[model_name]
    ckpt_dir = os.path.join(CHECKPOINT_ROOT, f"seed_{seed}", model_key)
    os.makedirs(ckpt_dir, exist_ok=True)

    print("\n" + "=" * 80)
    print(f"Seed: {seed}")
    print(f"Model: {model_name}")
    print(f"Train count: {len(X_train)}")
    print(f"Validation count: {len(X_val)}")
    print(f"Test count: {len(X_test)}")
    print("=" * 80)

    # Re-seed right before this run so every model at this seed starts from
    # the identical stochastic state (fair comparison within the seed).
    set_global_seed(seed)

    # Sanity checks
    assert len(X_train) == len(y_train), "Train image/mask count mismatch"
    assert len(X_val) == len(y_val), "Val image/mask count mismatch"
    assert len(X_test) == len(y_test), "Test image/mask count mismatch"
    assert X_train.shape[1:3] == (IMG_HEIGHT, IMG_WIDTH), "Unexpected input shape"

    train_dataset = create_augmented_dataset_with_preprocessing(
        X_train, y_train, batch_size=BATCH_SIZE, augment=True, shuffle=True)
    val_dataset = create_augmented_dataset_with_preprocessing(
        X_val, y_val, batch_size=BATCH_SIZE, augment=False, shuffle=False)

    class_weights = calculate_class_weights(y_train)  # deterministic, unused directly but kept for parity

    model, backbone = MODEL_BUILDERS[model_name](input_shape=(IMG_HEIGHT, IMG_WIDTH, 3))

    if tuple(model.output_shape) != (None, IMG_HEIGHT, IMG_WIDTH, 1):
        raise ValueError(f"Unexpected output shape: {model.output_shape}")

    # ---------------- PHASE 1: frozen backbone ----------------
    backbone.trainable = False
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss=aggressive_imbalance_loss,
        metrics=[dice_coefficient, iou_metric]
    )

    callbacks_phase1 = [
        PredictionMonitor(X_val, y_val),
        tf.keras.callbacks.EarlyStopping(monitor='val_dice_coefficient', patience=15,
                                          restore_best_weights=True, mode='max', verbose=1),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_dice_coefficient', factor=0.5,
                                              patience=5, min_lr=1e-6, mode='max', verbose=1),
        tf.keras.callbacks.ModelCheckpoint(
            os.path.join(ckpt_dir, f'{model_key}_phase1.weights.h5'),
            monitor='val_dice_coefficient', save_best_only=True,
            save_weights_only=True, mode='max', verbose=1)
    ]

    history_phase1 = model.fit(
        train_dataset, validation_data=val_dataset,
        epochs=PHASE1_EPOCHS, callbacks=callbacks_phase1, verbose=1
    )

    # ---------------- PHASE 2: fine-tune full model ----------------
    backbone.trainable = True
    for layer in backbone.layers[:-80]:
        layer.trainable = False
    for layer in model.layers:
        if isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = False

    optimizer_ft = tf.keras.optimizers.AdamW(
        learning_rate=5e-5, weight_decay=1e-5, beta_1=0.9, beta_2=0.999)

    model.compile(
        optimizer=optimizer_ft,
        loss=focal_dice_loss,
        metrics=[dice_coefficient, iou_metric, 'binary_accuracy',
                 tf.keras.metrics.Precision(name='precision'),
                 tf.keras.metrics.Recall(name='recall')]
    )

    callbacks_phase2 = [
        PredictionMonitor(X_val, y_val),
        tf.keras.callbacks.EarlyStopping(monitor='val_dice_coefficient', patience=PATIENCE,
                                          restore_best_weights=True, mode='max', verbose=1),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_dice_coefficient', factor=0.5,
                                              patience=8, min_lr=1e-7, mode='max', verbose=1),
        tf.keras.callbacks.ModelCheckpoint(
            os.path.join(ckpt_dir, f'{model_key}_final.weights.h5'),
            monitor='val_dice_coefficient', save_best_only=True,
            save_weights_only=True, mode='max', verbose=1)
    ]

    history_phase2 = model.fit(
        train_dataset, validation_data=val_dataset,
        epochs=EPOCHS, callbacks=callbacks_phase2, verbose=1,
        initial_epoch=len(history_phase1.history['loss'])
    )

    best_val_dice = max(history_phase2.history['val_dice_coefficient'])

    # ---------------- THRESHOLD SEARCH ON VALIDATION (never on test) ----------------
    X_val_pp = preprocess_test_data(X_val)
    y_val_pred_soft = model.predict(X_val_pp, batch_size=BATCH_SIZE, verbose=0)

    best_t, best_val_thresh_dice = 0.0, 0.0
    for t in THRESHOLD_RANGE:
        y_val_pred_bin = (y_val_pred_soft > t).astype(np.uint8)
        d = dice_numpy(y_val, y_val_pred_bin)
        if d > best_val_thresh_dice:
            best_val_thresh_dice = d
            best_t = t
    seg_threshold = float(best_t)

    # ---------------- TRAIN SET EVAL (frozen threshold) ----------------
    X_train_pp = preprocess_test_data(X_train)
    y_train_pred_soft = model.predict(X_train_pp, batch_size=BATCH_SIZE, verbose=0)
    y_train_pred_bin = (y_train_pred_soft > seg_threshold).astype(np.uint8)
    y_train_bin = (y_train > 0.5).astype(np.uint8)
    train_metrics = calculate_segmentation_metrics(y_train_bin, y_train_pred_bin)

    # ---------------- TEST SET EVAL (frozen threshold) ----------------
    X_test_pp = preprocess_test_data(X_test)
    y_test_pred_soft = model.predict(X_test_pp, batch_size=BATCH_SIZE, verbose=0)
    y_test_pred_bin = (y_test_pred_soft > seg_threshold).astype(np.uint8)
    y_test_bin = (y_test > 0.5).astype(np.uint8)
    test_metrics = calculate_segmentation_metrics(y_test_bin, y_test_pred_bin)

    print(f"\nSeed: {seed}")
    print(f"Model: {model_name}")
    print(f"Best validation Dice: {best_val_dice:.4f}")
    print(f"Selected threshold: {seg_threshold:.2f}")
    print(f"Test Dice: {test_metrics['dice']:.4f}")
    print(f"Test mIoU: {test_metrics['miou']:.4f}")
    print(f"Test Precision: {test_metrics['precision']:.4f}")
    print(f"Test Recall: {test_metrics['recall']:.4f}")

    # Save final weights explicitly (in addition to the ModelCheckpoint above)
    final_ckpt_path = os.path.join(ckpt_dir, f'{model_key}_seed{seed}_final.weights.h5')
    model.save_weights(final_ckpt_path)

    del model
    tf.keras.backend.clear_session()

    rows = [
        dict(Seed=seed, Model=model_name, Dataset='Train',
             Dice=train_metrics['dice'], Foreground_IoU=train_metrics['iou_foreground'],
             Background_IoU=train_metrics['iou_background'], mIoU=train_metrics['miou'],
             Precision=train_metrics['precision'], Recall=train_metrics['recall'],
             Threshold=seg_threshold),
        dict(Seed=seed, Model=model_name, Dataset='Test',
             Dice=test_metrics['dice'], Foreground_IoU=test_metrics['iou_foreground'],
             Background_IoU=test_metrics['iou_background'], mIoU=test_metrics['miou'],
             Precision=test_metrics['precision'], Recall=test_metrics['recall'],
             Threshold=seg_threshold),
    ]
    return rows, final_ckpt_path


# ==============================================================================
# MAIN DRIVER
# ==============================================================================

def parse_seed42_threshold(report_path):
    """Best-effort: pull 'Optimal threshold: X.XX' out of an original ablation
    report .txt file if present. Returns None if not found."""
    if not os.path.exists(report_path):
        return None
    try:
        text = open(report_path).read()
        m = re.search(r"Optimal threshold:\s*([0-9.]+)", text)
        return float(m.group(1)) if m else None
    except Exception:
        return None


def load_seed42_rows():
    """Load the existing seed-42 reference. Prefers the original per-model
    results.csv (has TRAIN+TEST); falls back to the TEST-only reference
    given in the prompt with TRAIN left as NaN."""
    rows = []
    for model_name in MODEL_ORDER:
        csv_path = SEED42_RESULTS_CSV_MAP.get(model_name)
        report_glob_key = MODEL_KEYS[model_name]
        threshold = None
        # try to find a matching ablation report for the threshold
        for cand in glob.glob(f"results_*{report_glob_key}*/*report*.txt") + glob.glob(f"results_*{report_glob_key}*/*.txt"):
            threshold = parse_seed42_threshold(cand)
            if threshold is not None:
                break

        if csv_path and os.path.exists(csv_path):
            df = pd.read_csv(csv_path)
            for _, r in df.iterrows():
                rows.append(dict(
                    Seed=42, Model=model_name, Dataset=r['Dataset'],
                    Dice=r['Dice'], Foreground_IoU=r['Foreground IoU'],
                    Background_IoU=r['Background IoU'], mIoU=r['mIoU'],
                    Precision=r['Precision'], Recall=r['Recall'],
                    Threshold=threshold if threshold is not None else np.nan
                ))
            print(f" Loaded seed-42 TRAIN+TEST rows for '{model_name}' from {csv_path}")
        else:
            ref = SEED42_TEST_REFERENCE[model_name]
            rows.append(dict(Seed=42, Model=model_name, Dataset='Test',
                              Dice=ref['Dice'], Foreground_IoU=ref['Foreground_IoU'],
                              Background_IoU=ref['Background_IoU'], mIoU=ref['mIoU'],
                              Precision=ref['Precision'], Recall=ref['Recall'],
                              Threshold=threshold if threshold is not None else np.nan))
            print(f" '{model_name}': no results.csv found at {csv_path} — using TEST-only "
                  f"reference from the prompt. TRAIN row for seed 42 is unavailable.")
    return rows


def main():
    configure_gpu()

    print("\n" + "=" * 80)
    print("LOADING DATASET (fixed split — identical for every seed & model)")
    print("=" * 80)
    X_train, y_train = load_data_raw(TRAIN_IMAGE_DIR, TRAIN_MASK_DIR)
    X_val, y_val = load_data_raw(VAL_IMAGE_DIR, VAL_MASK_DIR)
    X_test, y_test = load_data_raw(TEST_IMAGE_DIR, TEST_MASK_DIR)
    print(f"Train={len(X_train)}  Val={len(X_val)}  Test={len(X_test)}")

    seeds_to_run = list(SEEDS)
    if INCLUDE_SEED_42_RETRAIN:
        seeds_to_run = [42] + seeds_to_run

    run_registry = []   # Seed | Model | Status | Checkpoint | Notes
    all_rows = []

    for seed in seeds_to_run:
        for model_name in MODEL_ORDER:
            try:
                rows, ckpt_path = train_and_evaluate_one(
                    seed, model_name, X_train, y_train, X_val, y_val, X_test, y_test)
                all_rows.extend(rows)
                run_registry.append(dict(Seed=seed, Model=model_name, Status='SUCCESS',
                                          Checkpoint=ckpt_path, Notes=''))
            except Exception as e:
                print(f"\nFAILED:\nSeed = {seed}\nModel = {model_name}\nReason = {e}")
                run_registry.append(dict(Seed=seed, Model=model_name, Status='FAILED',
                                          Checkpoint='', Notes=str(e)))

    # Merge in the (non-retrained) seed-42 reference rows
    if not INCLUDE_SEED_42_RETRAIN:
        seed42_rows = load_seed42_rows()
        all_rows.extend(seed42_rows)
        for model_name in MODEL_ORDER:
            run_registry.append(dict(Seed=42, Model=model_name, Status='EXISTING_REFERENCE',
                                      Checkpoint='(pre-existing seed-42 run)', Notes=''))

    results_df = pd.DataFrame(all_rows)
    results_df = results_df[['Seed', 'Model', 'Dataset', 'Dice', 'Foreground_IoU',
                              'Background_IoU', 'mIoU', 'Precision', 'Recall', 'Threshold']]
    results_df.sort_values(['Seed', 'Model', 'Dataset'], inplace=True)
    results_path = os.path.join(OUTPUT_DIR, 'three_seed_ablation_results.csv')
    results_df.to_csv(results_path, index=False)
    print(f"\nSaved {results_path}")

    # ---------------- SUMMARY STATISTICS ----------------
    metric_cols = ['Dice', 'Foreground_IoU', 'Background_IoU', 'mIoU', 'Precision', 'Recall']
    summary_rows = []
    for model_name in MODEL_ORDER:
        for dataset in ['Train', 'Test']:
            sub = results_df[(results_df['Model'] == model_name) & (results_df['Dataset'] == dataset)]
            if sub.empty:
                continue
            row = dict(Model=model_name, Dataset=dataset)
            for m in metric_cols:
                row[f'{m}_Mean'] = sub[m].mean()
                row[f'{m}_STD'] = sub[m].std(ddof=1) if len(sub) > 1 else 0.0
            summary_rows.append(row)
    summary_df = pd.DataFrame(summary_rows)
    summary_path = os.path.join(OUTPUT_DIR, 'three_seed_ablation_summary.csv')
    summary_df.to_csv(summary_path, index=False)
    print(f"Saved {summary_path}")

    # ---------------- MAIN PAPER TABLE (TEST only) ----------------
    def fmt(mean, std):
        return f"{mean:.4f} ± {std:.4f}"

    table_rows = []
    for model_name in MODEL_ORDER:
        row = summary_df[(summary_df['Model'] == model_name) & (summary_df['Dataset'] == 'Test')]
        if row.empty:
            continue
        row = row.iloc[0]
        table_rows.append({
            'Model': model_name,
            'Dice (mean ± std)': fmt(row['Dice_Mean'], row['Dice_STD']),
            'Foreground IoU (mean ± std)': fmt(row['Foreground_IoU_Mean'], row['Foreground_IoU_STD']),
            'mIoU (mean ± std)': fmt(row['mIoU_Mean'], row['mIoU_STD']),
            'Precision (mean ± std)': fmt(row['Precision_Mean'], row['Precision_STD']),
            'Recall (mean ± std)': fmt(row['Recall_Mean'], row['Recall_STD']),
        })
    test_summary_df = pd.DataFrame(table_rows)
    test_summary_path = os.path.join(OUTPUT_DIR, 'three_seed_test_summary.csv')
    test_summary_df.to_csv(test_summary_path, index=False)
    print(f"Saved {test_summary_path}\n")
    print(test_summary_df.to_string(index=False))

    # ---------------- COMPARISON WITH SEED 42 ----------------
    print("\n" + "=" * 80)
    print("COMPARISON WITH SEED 42 (was seed 42 unusually high or low?)")
    print("=" * 80)
    comp_rows = []
    for model_name in MODEL_ORDER:
        seed42_row = results_df[(results_df['Model'] == model_name) &
                                 (results_df['Seed'] == 42) & (results_df['Dataset'] == 'Test')]
        summary_row = summary_df[(summary_df['Model'] == model_name) & (summary_df['Dataset'] == 'Test')]
        if seed42_row.empty or summary_row.empty:
            continue
        seed42_row = seed42_row.iloc[0]
        summary_row = summary_row.iloc[0]
        for m in metric_cols:
            comp_rows.append(dict(
                Model=model_name, Metric=m,
                Seed42_Value=seed42_row[m],
                ThreeSeed_Mean=summary_row[f'{m}_Mean'],
                Difference=seed42_row[m] - summary_row[f'{m}_Mean']
            ))
    comp_df = pd.DataFrame(comp_rows)
    comp_path = os.path.join(OUTPUT_DIR, 'seed42_vs_3seed_comparison.csv')
    comp_df.to_csv(comp_path, index=False)
    print(comp_df.to_string(index=False))
    print(f"\nSaved {comp_path}")

    # ---------------- STATISTICAL COMPARISON (no fabricated significance) ----------------
    print("\n" + "=" * 80)
    print("STATISTICAL COMPARISON (descriptive only — n=3 seeds has very limited power)")
    print("=" * 80)
    test_only = results_df[results_df['Dataset'] == 'Test']
    pivot = test_only.pivot_table(index='Seed', columns='Model', values='Dice')
    pivot = pivot.dropna()  # only seeds present for all models
    if len(pivot) >= 2:
        for i in range(len(MODEL_ORDER)):
            for j in range(i + 1, len(MODEL_ORDER)):
                a, b = MODEL_ORDER[i], MODEL_ORDER[j]
                if a not in pivot.columns or b not in pivot.columns:
                    continue
                diff = pivot[a] - pivot[b]
                print(f"{a}  vs  {b}")
                print(f"  Paired Dice differences across seeds {list(pivot.index)}: {diff.values}")
                print(f"  Mean diff: {diff.mean():.5f}   STD diff: {diff.std(ddof=1) if len(diff) > 1 else 0.0:.5f}")
                if len(diff) >= 2 and diff.std(ddof=1) > 0:
                    t_stat, p_val = stats.ttest_rel(pivot[a], pivot[b])
                    print(f"  Paired t-test: t={t_stat:.3f}, p={p_val:.3f} "
                          f"(NOTE: n={len(diff)} seeds — this is severely underpowered; "
                          f"do not treat this p-value as evidence of significance)")
                print()
    else:
        print("Fewer than 2 seeds with results for all models — skipping paired comparison.")

    # ---------------- VISUALIZATION ----------------
    make_plots(results_df)

    # ---------------- RUN REGISTRY ----------------
    registry_df = pd.DataFrame(run_registry)
    registry_path = os.path.join(OUTPUT_DIR, 'run_registry.csv')
    registry_df.to_csv(registry_path, index=False)

    trained_runs = registry_df[registry_df['Status'].isin(['SUCCESS', 'FAILED'])]
    n_success = (trained_runs['Status'] == 'SUCCESS').sum()
    n_failed = (trained_runs['Status'] == 'FAILED').sum()
    n_total = len(trained_runs) if len(trained_runs) > 0 else (len(seeds_to_run) * len(MODEL_ORDER))

    print("\n" + "=" * 80)
    print(f"Successful runs: {n_success} / {n_total}")
    print(f"Failed runs: {n_failed} / {n_total}")
    print("=" * 80)

    print("\nFINAL TEST TABLE")
    print(test_summary_df.to_string(index=False))

    print("\n3-SEED REPRODUCIBILITY EXPERIMENT COMPLETE")


def make_plots(results_df):
    test_only = results_df[results_df['Dataset'] == 'Test'].copy()

    metrics_to_plot = [('Dice', 'Test Dice by Model and Seed'),
                        ('mIoU', 'Test mIoU by Model and Seed'),
                        ('Recall', 'Test Recall by Model and Seed'),
                        ('Precision', 'Test Precision by Model and Seed')]

    seeds = sorted(test_only['Seed'].unique())
    x = np.arange(len(MODEL_ORDER))
    width = 0.8 / max(len(seeds), 1)

    for metric, title in metrics_to_plot:
        fig, ax = plt.subplots(figsize=(11, 6))
        for i, seed in enumerate(seeds):
            vals = []
            for model_name in MODEL_ORDER:
                v = test_only[(test_only['Seed'] == seed) & (test_only['Model'] == model_name)][metric]
                vals.append(v.values[0] if len(v) else np.nan)
            ax.bar(x + i * width, vals, width, label=f'Seed {seed}')
        ax.set_xticks(x + width * (len(seeds) - 1) / 2)
        ax.set_xticklabels([m.replace('EfficientNetV2-B3', 'EffNetV2-B3') for m in MODEL_ORDER],
                            rotation=20, ha='right')
        ax.set_ylabel(metric)
        ax.set_title(title)
        lo = max(0.0, test_only[metric].min() - 0.02)
        hi = min(1.0, test_only[metric].max() + 0.02)
        ax.set_ylim(lo, hi)  # not truncated at 0 for readability, but not misleadingly narrow either
        ax.legend()
        ax.grid(True, alpha=0.3, axis='y')
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR, f'test_{metric.lower()}_by_model_seed.png'), dpi=200)
        plt.close(fig)

    # Mean ± std comparison
    fig, ax = plt.subplots(figsize=(11, 6))
    means, stds = [], []
    for model_name in MODEL_ORDER:
        vals = test_only[test_only['Model'] == model_name]['Dice']
        means.append(vals.mean())
        stds.append(vals.std(ddof=1) if len(vals) > 1 else 0.0)
    ax.bar(x, means, yerr=stds, capsize=6, color='#3498db', edgecolor='black')
    ax.set_xticks(x)
    ax.set_xticklabels([m.replace('EfficientNetV2-B3', 'EffNetV2-B3') for m in MODEL_ORDER],
                        rotation=20, ha='right')
    ax.set_ylabel('Test Dice')
    ax.set_ylim(0, 1.0)
    ax.set_title('Test Dice — Mean ± STD across seeds')
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'test_dice_mean_std_comparison.png'), dpi=200)
    plt.close(fig)

    print(f"\nSaved plots to {OUTPUT_DIR}/")


if __name__ == '__main__':
    main()

 GPU detected: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
 Mixed precision enabled

LOADING DATASET (fixed split — identical for every seed & model)
Loading RAW data from: /content/training/training/Training_original
Found 556 images
Successfully loaded 556 images
Loading RAW data from: /content/val/val/Original
Found 208 images
Successfully loaded 100 images
Loading RAW data from: /content/Testing/Testing/Original
Found 200 images
Successfully loaded 100 images
Train=556  Val=100  Test=100

Seed: 123
Model: EfficientNetV2-B3 Baseline
Train count: 556
Validation count: 100
Test count: 100
 Global seed set to 123 (python random / numpy / tensorflow)
52606240/52606240 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step
Epoch 1/40
139/139 ━━━━━━━━━━━━━━━━━━━━ 0s 108ms/step - dice_coefficient: 0.4457 - iou_metric: 0.3179 - loss: 4.4144   [Epoch 0] max=1.0000, mean=0.0966, fg_pixels=100,789/1,048,576 (9.612%)

Epoch 1: val_dice_coefficient improved from None to 0.33372, saving model to